In [25]:
import pandas as pd
import numpy as np
import zipfile
from pathlib import Path
from statsmodels.tsa.arima.model import ARIMA
from sklearn.linear_model import LinearRegression
import warnings

warnings.filterwarnings("ignore")

# Extraccción

## Precipitación

### Consolidación Base Precipitación Tolima

In [26]:
import pandas as pd
import numpy as np
import zipfile
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

ruta = Path("Tolima_Precipitación.zip")

if ruta.suffix == ".zip":
    carpeta = ruta.with_suffix("")
    carpeta.mkdir(exist_ok=True)

    with zipfile.ZipFile(ruta, "r") as zip_ref:
        zip_ref.extractall(carpeta)
else:
    carpeta = ruta

archivos = list(carpeta.glob("*.xlsx"))

base = None

for archivo in archivos:
    print("Procesando:", archivo.name)

    nombre_estacion = pd.read_excel(
        archivo,
        header=None,
        nrows=2
    ).iloc[1, 1]

    if pd.isna(nombre_estacion):
        nombre_estacion = archivo.stem

    nombre_estacion = str(nombre_estacion)

    df = pd.read_excel(
        archivo,
        sheet_name="Datos",
        header=None,
        skiprows=7
    )

    df = df[[0, 2]]
    df.columns = ["fecha", nombre_estacion]

    df["fecha"] = pd.to_datetime(df["fecha"], errors="coerce")

    df[nombre_estacion] = (
        df[nombre_estacion]
        .astype(str)
        .str.replace(",", ".", regex=False)
        .str.replace(" ", "", regex=False)
        .str.replace("-", "", regex=False)
    )

    df[nombre_estacion] = pd.to_numeric(
        df[nombre_estacion],
        errors="coerce"
    )

    df = df.dropna(subset=["fecha"])

    if base is None:
        base = df
    else:
        base = pd.merge(
            base,
            df,
            on="fecha",
            how="outer"
        )

base = base.sort_values("fecha").reset_index(drop=True)

#Dato solo desde 2010
base = base[
    base["fecha"] >= pd.Timestamp("2010-01-01")
].reset_index(drop=True)

original = base.copy()

print("Base consolidada")
print(
    "Fechas:",
    base["fecha"].min(),
    "hasta",
    base["fecha"].max()
)

original = base.copy()

print("Base consolidada")

Procesando: 21135020.xlsx
Procesando: 21135030.xlsx
Procesando: 21185030.xlsx
Procesando: 21185080.xlsx
Procesando: 21215080.xlsx
Procesando: 21245040.xlsx
Procesando: 21245140.xlsx
Procesando: 22055020.xlsx
Procesando: 22065040.xlsx
Procesando: 23025040.xlsx
Base consolidada
Fechas: 2010-01-01 00:00:00 hasta 2025-12-31 00:00:00
Base consolidada


### Prueba de estacionariedad y estacionalidad

In [27]:
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.seasonal import seasonal_decompose

resumen_pruebas = []

for columna in base.columns:

    if columna == "fecha":
        continue

    serie = base[["fecha", columna]].dropna()
    serie = serie.set_index("fecha")[columna]

    if len(serie) < 24:
        resumen_pruebas.append({
            "estacion": columna,
            "n_observaciones": len(serie),
            "ADF_p_value": np.nan,
            "es_estacionaria": "No evaluable",
            "observacion": "Muy pocos datos"
        })
        continue

    resultado_adf = adfuller(serie)

    p_value = resultado_adf[1]

    es_estacionaria = "Sí" if p_value < 0.05 else "No"

    resumen_pruebas.append({
        "estacion": columna,
        "n_observaciones": len(serie),
        "ADF_p_value": p_value,
        "es_estacionaria": es_estacionaria,
        "observacion": "Evaluada"
    })

resumen_pruebas = pd.DataFrame(resumen_pruebas)

resumen_pruebas

,estacion,n_observaciones,ADF_p_value,es_estacionaria,observacion
0,JABALCON [21135020],5676,3.037309e-29,Sí,Evaluada
1,ANCHIQUE [21135030],5587,2.743171e-30,Sí,Evaluada
2,GUAMO [21185030],5665,0.000000e+00,Sí,Evaluada
3,VALLE DE SAN JUAN [21185080],5836,0.000000e+00,Sí,Evaluada
4,CHICORAL [21215080],5401,3.060139e-29,Sí,Evaluada
5,AEROPUERTO PERALES [21245040],5492,2.395498e-26,Sí,Evaluada
6,SANTA ISABEL [21245140],5762,1.189650e-18,Sí,Evaluada
7,MESA DE POLE [22055020],5726,7.612051e-16,Sí,Evaluada
8,SAN ANTONIO QUINTA [22065040],5775,9.996504e-28,Sí,Evaluada
9,ALBANIA - AUT [23025040],5835,1.972213e-16,Sí,Evaluada


### ARIMA para vacios


In [28]:
from statsmodels.tsa.arima.model import ARIMA
!pip install pmdarima
from pmdarima import auto_arima

def imputar_arima(df, columna):

    datos = df[["fecha", columna]].copy()
    datos = datos.sort_values("fecha")
    datos = datos.set_index("fecha")

    y = pd.to_numeric(
        datos[columna],
        errors="coerce"
    )

    # Guardar ubicación de vacíos originales
    mask_na_original = y.isna()

    # Si hay pocos datos usar interpolación
    if y.notna().sum() < 30:

        y_final = (
            y.interpolate("linear")
            .ffill()
            .bfill()
        )

        return (
            y_final.reset_index(drop=True),
            "Interpolación (pocos datos)"
        )

    # Interpolación temporal preliminar
    # necesaria para entrenar ARIMA
    y_temp = (
        y.interpolate("linear")
        .ffill()
        .bfill()
    )

    try:

        # Selección automática de p,d,q
        modelo_auto = auto_arima(
            y_temp,
            seasonal=False,
            d=0,   # ADF mostró estacionariedad
            trace=False,
            suppress_warnings=True
        )

        p,d,q = modelo_auto.order

        modelo = ARIMA(
            y_temp,
            order=(p,d,q)
        )

        ajuste = modelo.fit()

        predicciones = ajuste.predict(
            start=0,
            end=len(y_temp)-1
        )

        y_final = y.copy()

        # Solo reemplazar vacíos originales
        y_final.loc[
            mask_na_original
        ] = predicciones.loc[
            mask_na_original
        ]

        return (
            y_final.reset_index(drop=True),
            f"ARIMA{(p,d,q)}"
        )

    except Exception as e:

        y_final = (
            y.interpolate("linear")
            .ffill()
            .bfill()
        )

        return (
            y_final.reset_index(drop=True),
            f"Interpolación por error: {e}"
        )

In [29]:
def seleccionar_mejor_arima(serie):

    mejor_aic = np.inf
    mejor_orden = None

    ordenes = [
        (1, 0, 0),
        (0, 0, 1),
        (1, 0, 1),
        (2, 0, 0),
        (0, 0, 2),
        (2, 0, 1),
        (1, 0, 2)
    ]

    for orden in ordenes:

        try:
            modelo = ARIMA(
                serie,
                order=orden
            )

            resultado = modelo.fit(
                method_kwargs={"maxiter": 50}
            )

            if resultado.aic < mejor_aic:
                mejor_aic = resultado.aic
                mejor_orden = orden

        except:
            continue

    if mejor_orden is None:
        mejor_orden = (1, 0, 0)

    return mejor_orden

In [30]:
def imputar_arima(df, columna):

    datos = df[["fecha", columna]].copy()
    datos = datos.sort_values("fecha")

    datos = datos.set_index("fecha")
    datos.index = pd.to_datetime(datos.index)

    y = pd.to_numeric(
        datos[columna],
        errors="coerce"
    )

    mask_na_original = y.isna()

    # Si hay pocos datos, usar interpolación temporal
    if y.notna().sum() < 30:

        y_final = (
            y
            .interpolate(method="time")
            .ffill()
            .bfill()
        )

        return (
            y_final.reset_index(drop=True),
            "Interpolación temporal por pocos datos"
        )

    # Interpolación preliminar para que ARIMA pueda entrenarse
    y_temp = (
        y
        .interpolate(method="time")
        .ffill()
        .bfill()
    )

    try:
        orden = seleccionar_mejor_arima(y_temp)

        modelo = ARIMA(
            y_temp,
            order=orden
        )

        ajuste = modelo.fit(
            method_kwargs={"maxiter": 50}
        )

        predicciones = ajuste.predict(
            start=0,
            end=len(y_temp) - 1
        )

        y_final = y.copy()

        # Solo reemplaza los datos que originalmente estaban vacíos
        y_final.loc[mask_na_original] = predicciones.loc[mask_na_original]

        # Respaldo final: si queda algún vacío, se completa
        y_final = (
            y_final
            .interpolate(method="time")
            .ffill()
            .bfill()
        )

        return (
            y_final.reset_index(drop=True),
            f"ARIMA{orden} + respaldo temporal"
        )

    except Exception as e:

        y_final = (
            y
            .interpolate(method="time")
            .ffill()
            .bfill()
        )

        return (
            y_final.reset_index(drop=True),
            f"Interpolación temporal porque ARIMA falló: {e}"
        )

In [31]:
completado = base.copy()

marca_imputacion = pd.DataFrame()
marca_imputacion["fecha"] = base["fecha"]

resumen_modelos = []

for columna in base.columns:

    if columna == "fecha":
        continue

    print("Imputando:", columna)

    marca_imputacion[columna] = base[columna].isna()

    serie_completa, metodo = imputar_arima(
        base,
        columna
    )

    completado[columna] = serie_completa

    resumen_modelos.append({
        "estacion": columna,
        "modelo_usado": metodo,
        "observaciones_originales": base[columna].notna().sum(),
        "valores_imputados": base[columna].isna().sum(),
        "porcentaje_imputado": round(
            100 * base[columna].isna().sum() / len(base),
            2
        ),
        "faltantes_finales": completado[columna].isna().sum()
    })

resumen_modelos = pd.DataFrame(resumen_modelos)

print("Imputación terminada")

Imputando: JABALCON [21135020]
Imputando: ANCHIQUE [21135030]
Imputando: GUAMO [21185030]
Imputando: VALLE DE SAN JUAN [21185080]
Imputando: CHICORAL [21215080]
Imputando: AEROPUERTO PERALES [21245040]
Imputando: SANTA ISABEL [21245140]
Imputando: MESA DE POLE [22055020]
Imputando: SAN ANTONIO QUINTA [22065040]
Imputando: ALBANIA - AUT [23025040]
Imputación terminada


In [32]:
faltantes_finales = completado.isna().sum()

faltantes_finales

fecha                            0
JABALCON [21135020]              0
ANCHIQUE [21135030]              0
GUAMO [21185030]                 0
VALLE DE SAN JUAN [21185080]     0
CHICORAL [21215080]              0
AEROPUERTO PERALES [21245040]    0
SANTA ISABEL [21245140]          0
MESA DE POLE [22055020]          0
SAN ANTONIO QUINTA [22065040]    0
ALBANIA - AUT [23025040]         0
dtype: int64

### Exportar Excel

In [33]:
with pd.ExcelWriter(
    "Tolima_Precipitación.xlsx",
    engine="openpyxl"
) as writer:

    completado.to_excel(
        writer,
        sheet_name="Completado",
        index=False
    )

    original.to_excel(
        writer,
        sheet_name="Original",
        index=False
    )

    marca_imputacion.to_excel(
        writer,
        sheet_name="Marca_imputacion",
        index=False
    )

    resumen_pruebas.to_excel(
        writer,
        sheet_name="Pruebas_ADF",
        index=False
    )

    resumen_modelos.to_excel(
        writer,
        sheet_name="Resumen_modelos",
        index=False
    )

print("Archivo creado: Tolima_Precipitación.xlsx")

Archivo creado: Tolima_Precipitación.xlsx


## Temperatura Máxima

### Consolidación Base Precipitación Tolima

In [34]:
import pandas as pd
import numpy as np
import zipfile
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

ruta = Path("Tolima_Precipitación.zip")

if ruta.suffix == ".zip":
    carpeta = ruta.with_suffix("")
    carpeta.mkdir(exist_ok=True)

    with zipfile.ZipFile(ruta, "r") as zip_ref:
        zip_ref.extractall(carpeta)
else:
    carpeta = ruta

archivos = list(carpeta.glob("*.xlsx"))

base = None

for archivo in archivos:
    print("Procesando:", archivo.name)

    nombre_estacion = pd.read_excel(
        archivo,
        header=None,
        nrows=2
    ).iloc[1, 1]

    if pd.isna(nombre_estacion):
        nombre_estacion = archivo.stem

    nombre_estacion = str(nombre_estacion)

    df = pd.read_excel(
        archivo,
        sheet_name="Datos",
        header=None,
        skiprows=7
    )

    df = df[[0, 2]]
    df.columns = ["fecha", nombre_estacion]

    df["fecha"] = pd.to_datetime(df["fecha"], errors="coerce")

    df[nombre_estacion] = (
        df[nombre_estacion]
        .astype(str)
        .str.replace(",", ".", regex=False)
        .str.replace(" ", "", regex=False)
        .str.replace("-", "", regex=False)
    )

    df[nombre_estacion] = pd.to_numeric(
        df[nombre_estacion],
        errors="coerce"
    )

    df = df.dropna(subset=["fecha"])

    if base is None:
        base = df
    else:
        base = pd.merge(
            base,
            df,
            on="fecha",
            how="outer"
        )

base = base.sort_values("fecha").reset_index(drop=True)

#Dato solo desde 2010
base = base[
    base["fecha"] >= pd.Timestamp("2010-01-01")
].reset_index(drop=True)

original = base.copy()

print("Base consolidada")
print(
    "Fechas:",
    base["fecha"].min(),
    "hasta",
    base["fecha"].max()
)

original = base.copy()

print("Base consolidada")

Procesando: 21135020.xlsx
Procesando: 21135030.xlsx
Procesando: 21185030.xlsx
Procesando: 21185080.xlsx
Procesando: 21215080.xlsx
Procesando: 21245040.xlsx
Procesando: 21245140.xlsx
Procesando: 22055020.xlsx
Procesando: 22065040.xlsx
Procesando: 23025040.xlsx
Base consolidada
Fechas: 2010-01-01 00:00:00 hasta 2025-12-31 00:00:00
Base consolidada


### Prueba de estacionariedad y estacionalidad

In [35]:
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.seasonal import seasonal_decompose

resumen_pruebas = []

for columna in base.columns:

    if columna == "fecha":
        continue

    serie = base[["fecha", columna]].dropna()
    serie = serie.set_index("fecha")[columna]

    if len(serie) < 24:
        resumen_pruebas.append({
            "estacion": columna,
            "n_observaciones": len(serie),
            "ADF_p_value": np.nan,
            "es_estacionaria": "No evaluable",
            "observacion": "Muy pocos datos"
        })
        continue

    resultado_adf = adfuller(serie)

    p_value = resultado_adf[1]

    es_estacionaria = "Sí" if p_value < 0.05 else "No"

    resumen_pruebas.append({
        "estacion": columna,
        "n_observaciones": len(serie),
        "ADF_p_value": p_value,
        "es_estacionaria": es_estacionaria,
        "observacion": "Evaluada"
    })

resumen_pruebas = pd.DataFrame(resumen_pruebas)

resumen_pruebas

,estacion,n_observaciones,ADF_p_value,es_estacionaria,observacion
0,JABALCON [21135020],5676,3.037309e-29,Sí,Evaluada
1,ANCHIQUE [21135030],5587,2.743171e-30,Sí,Evaluada
2,GUAMO [21185030],5665,0.000000e+00,Sí,Evaluada
3,VALLE DE SAN JUAN [21185080],5836,0.000000e+00,Sí,Evaluada
4,CHICORAL [21215080],5401,3.060139e-29,Sí,Evaluada
5,AEROPUERTO PERALES [21245040],5492,2.395498e-26,Sí,Evaluada
6,SANTA ISABEL [21245140],5762,1.189650e-18,Sí,Evaluada
7,MESA DE POLE [22055020],5726,7.612051e-16,Sí,Evaluada
8,SAN ANTONIO QUINTA [22065040],5775,9.996504e-28,Sí,Evaluada
9,ALBANIA - AUT [23025040],5835,1.972213e-16,Sí,Evaluada


In [36]:
completado = base.copy()

marca_imputacion = pd.DataFrame()
marca_imputacion["fecha"] = base["fecha"]

resumen_modelos = []

for columna in base.columns:

    if columna == "fecha":
        continue

    print("Imputando:", columna)

    marca_imputacion[columna] = base[columna].isna()

    serie_completa, metodo = imputar_arima(
        base,
        columna
    )

    completado[columna] = serie_completa

    resumen_modelos.append({
        "estacion": columna,
        "modelo_usado": metodo,
        "observaciones_originales": base[columna].notna().sum(),
        "valores_imputados": base[columna].isna().sum(),
        "porcentaje_imputado": round(
            100 * base[columna].isna().sum() / len(base),
            2
        ),
        "faltantes_finales": completado[columna].isna().sum()
    })

resumen_modelos = pd.DataFrame(resumen_modelos)

print("Imputación terminada")

Imputando: JABALCON [21135020]
Imputando: ANCHIQUE [21135030]
Imputando: GUAMO [21185030]
Imputando: VALLE DE SAN JUAN [21185080]
Imputando: CHICORAL [21215080]
Imputando: AEROPUERTO PERALES [21245040]
Imputando: SANTA ISABEL [21245140]
Imputando: MESA DE POLE [22055020]
Imputando: SAN ANTONIO QUINTA [22065040]
Imputando: ALBANIA - AUT [23025040]
Imputación terminada


In [37]:
faltantes_finales = completado.isna().sum()

faltantes_finales

fecha                            0
JABALCON [21135020]              0
ANCHIQUE [21135030]              0
GUAMO [21185030]                 0
VALLE DE SAN JUAN [21185080]     0
CHICORAL [21215080]              0
AEROPUERTO PERALES [21245040]    0
SANTA ISABEL [21245140]          0
MESA DE POLE [22055020]          0
SAN ANTONIO QUINTA [22065040]    0
ALBANIA - AUT [23025040]         0
dtype: int64

### Exportar Excel

In [38]:
with pd.ExcelWriter(
    "Tolima_TempMax.xlsx",
    engine="openpyxl"
) as writer:

    completado.to_excel(
        writer,
        sheet_name="Completado",
        index=False
    )

    original.to_excel(
        writer,
        sheet_name="Original",
        index=False
    )

    marca_imputacion.to_excel(
        writer,
        sheet_name="Marca_imputacion",
        index=False
    )

    resumen_pruebas.to_excel(
        writer,
        sheet_name="Pruebas_ADF",
        index=False
    )

    resumen_modelos.to_excel(
        writer,
        sheet_name="Resumen_modelos",
        index=False
    )

print("Archivo creado: Tolima_TempMax.xlsx")

Archivo creado: Tolima_TempMax.xlsx


## Temperatura Mínima

### Consolidación Base Precipitación Tolima

In [39]:
import pandas as pd
import numpy as np
import zipfile
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

ruta = Path("Tolima_TempMin.zip")

if ruta.suffix == ".zip":
    carpeta = ruta.with_suffix("")
    carpeta.mkdir(exist_ok=True)

    with zipfile.ZipFile(ruta, "r") as zip_ref:
        zip_ref.extractall(carpeta)
else:
    carpeta = ruta

archivos = list(carpeta.glob("*.xlsx"))

base = None

for archivo in archivos:
    print("Procesando:", archivo.name)

    nombre_estacion = pd.read_excel(
        archivo,
        header=None,
        nrows=2
    ).iloc[1, 1]

    if pd.isna(nombre_estacion):
        nombre_estacion = archivo.stem

    nombre_estacion = str(nombre_estacion)

    df = pd.read_excel(
        archivo,
        sheet_name="Datos",
        header=None,
        skiprows=7
    )

    df = df[[0, 2]]
    df.columns = ["fecha", nombre_estacion]

    df["fecha"] = pd.to_datetime(df["fecha"], errors="coerce")

    df[nombre_estacion] = (
        df[nombre_estacion]
        .astype(str)
        .str.replace(",", ".", regex=False)
        .str.replace(" ", "", regex=False)
        .str.replace("-", "", regex=False)
    )

    df[nombre_estacion] = pd.to_numeric(
        df[nombre_estacion],
        errors="coerce"
    )

    df = df.dropna(subset=["fecha"])

    if base is None:
        base = df
    else:
        base = pd.merge(
            base,
            df,
            on="fecha",
            how="outer"
        )

base = base.sort_values("fecha").reset_index(drop=True)

#Dato solo desde 2010
base = base[
    base["fecha"] >= pd.Timestamp("2010-01-01")
].reset_index(drop=True)

original = base.copy()

print("Base consolidada")
print(
    "Fechas:",
    base["fecha"].min(),
    "hasta",
    base["fecha"].max()
)

original = base.copy()

print("Base consolidada")

Procesando: 21135020.xlsx
Procesando: 21135030.xlsx
Procesando: 21185030.xlsx
Procesando: 21185080.xlsx
Procesando: 21215080.xlsx
Procesando: 21245040.xlsx
Procesando: 21245140.xlsx
Procesando: 22055020.xlsx
Procesando: 22065040.xlsx
Procesando: 23025040.xlsx
Base consolidada
Fechas: 2010-01-01 00:00:00 hasta 2025-12-31 00:00:00
Base consolidada


### Prueba de estacionariedad y estacionalidad

In [40]:
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.seasonal import seasonal_decompose

resumen_pruebas = []

for columna in base.columns:

    if columna == "fecha":
        continue

    serie = base[["fecha", columna]].dropna()
    serie = serie.set_index("fecha")[columna]

    if len(serie) < 24:
        resumen_pruebas.append({
            "estacion": columna,
            "n_observaciones": len(serie),
            "ADF_p_value": np.nan,
            "es_estacionaria": "No evaluable",
            "observacion": "Muy pocos datos"
        })
        continue

    resultado_adf = adfuller(serie)

    p_value = resultado_adf[1]

    es_estacionaria = "Sí" if p_value < 0.05 else "No"

    resumen_pruebas.append({
        "estacion": columna,
        "n_observaciones": len(serie),
        "ADF_p_value": p_value,
        "es_estacionaria": es_estacionaria,
        "observacion": "Evaluada"
    })

resumen_pruebas = pd.DataFrame(resumen_pruebas)

resumen_pruebas

,estacion,n_observaciones,ADF_p_value,es_estacionaria,observacion
0,JABALCON [21135020],4421,4.198581e-06,Sí,Evaluada
1,ANCHIQUE [21135030],5471,1.000032e-07,Sí,Evaluada
2,GUAMO [21185030],5155,6.922417e-07,Sí,Evaluada
3,VALLE DE SAN JUAN [21185080],5579,1.215147e-07,Sí,Evaluada
4,CHICORAL [21215080],4313,1.468148e-18,Sí,Evaluada
5,AEROPUERTO PERALES [21245040],5418,2.288492e-05,Sí,Evaluada
6,SANTA ISABEL [21245140],4164,5.261262e-07,Sí,Evaluada
7,MESA DE POLE [22055020],5717,2.000228e-25,Sí,Evaluada
8,SAN ANTONIO QUINTA [22065040],5539,2.692347e-04,Sí,Evaluada
9,ALBANIA - AUT [23025040],5785,2.427831e-07,Sí,Evaluada


In [41]:
completado = base.copy()

marca_imputacion = pd.DataFrame()
marca_imputacion["fecha"] = base["fecha"]

resumen_modelos = []

for columna in base.columns:

    if columna == "fecha":
        continue

    print("Imputando:", columna)

    marca_imputacion[columna] = base[columna].isna()

    serie_completa, metodo = imputar_arima(
        base,
        columna
    )

    completado[columna] = serie_completa

    resumen_modelos.append({
        "estacion": columna,
        "modelo_usado": metodo,
        "observaciones_originales": base[columna].notna().sum(),
        "valores_imputados": base[columna].isna().sum(),
        "porcentaje_imputado": round(
            100 * base[columna].isna().sum() / len(base),
            2
        ),
        "faltantes_finales": completado[columna].isna().sum()
    })

resumen_modelos = pd.DataFrame(resumen_modelos)

print("Imputación terminada")

Imputando: JABALCON [21135020]
Imputando: ANCHIQUE [21135030]
Imputando: GUAMO [21185030]
Imputando: VALLE DE SAN JUAN [21185080]
Imputando: CHICORAL [21215080]
Imputando: AEROPUERTO PERALES [21245040]
Imputando: SANTA ISABEL [21245140]
Imputando: MESA DE POLE [22055020]
Imputando: SAN ANTONIO QUINTA [22065040]
Imputando: ALBANIA - AUT [23025040]
Imputación terminada


In [42]:
faltantes_finales = completado.isna().sum()

faltantes_finales

fecha                            0
JABALCON [21135020]              0
ANCHIQUE [21135030]              0
GUAMO [21185030]                 0
VALLE DE SAN JUAN [21185080]     0
CHICORAL [21215080]              0
AEROPUERTO PERALES [21245040]    0
SANTA ISABEL [21245140]          0
MESA DE POLE [22055020]          0
SAN ANTONIO QUINTA [22065040]    0
ALBANIA - AUT [23025040]         0
dtype: int64

### Exportar Excel

In [43]:
with pd.ExcelWriter(
    "Tolima_TempMin.xlsx",
    engine="openpyxl"
) as writer:

    completado.to_excel(
        writer,
        sheet_name="Completado",
        index=False
    )

    original.to_excel(
        writer,
        sheet_name="Original",
        index=False
    )

    marca_imputacion.to_excel(
        writer,
        sheet_name="Marca_imputacion",
        index=False
    )

    resumen_pruebas.to_excel(
        writer,
        sheet_name="Pruebas_ADF",
        index=False
    )

    resumen_modelos.to_excel(
        writer,
        sheet_name="Resumen_modelos",
        index=False
    )

print("Archivo creado: Tolima_TempMin.xlsx")

Archivo creado: Tolima_TempMin.xlsx


# Cosolidaación Capa A

## Promediar estaciones

In [44]:
archivos = {
    "Precipitacion": "Tolima_Precipitación.xlsx",
    "TempMax": "Tolima_TempMax.xlsx",
    "TempMin": "Tolima_TempMin.xlsx"
}

In [45]:
#CALCULAR PROMEDIOS
def calcular_promedio_diario(df, nombre_variable):

    df = df.copy()

    # Normalizar nombre de fecha
    if "fecha" in df.columns:
        col_fecha = "fecha"
    elif "date" in df.columns:
        col_fecha = "date"
    else:
        col_fecha = df.columns[0]

    df[col_fecha] = pd.to_datetime(
        df[col_fecha],
        errors="coerce"
    )

    columnas_valores = [
        col for col in df.columns
        if col != col_fecha
    ]

    for col in columnas_valores:
        df[col] = pd.to_numeric(
            df[col],
            errors="coerce"
        )

    promedio = pd.DataFrame()
    promedio["date"] = df[col_fecha]
    promedio[nombre_variable] = df[columnas_valores].mean(axis=1)

    return promedio

In [46]:
#Leer hojas de completado
hojas_completado = {}
promedios = []

for nombre, archivo in archivos.items():

    df = pd.read_excel(
        archivo,
        sheet_name="Completado"
    )

    hojas_completado[nombre] = df

    promedio = calcular_promedio_diario(
        df,
        nombre
    )

    promedios.append(promedio)

In [47]:
#Unir promedioas
promedios_final = promedios[0]

for promedio in promedios[1:]:
    promedios_final = promedios_final.merge(
        promedio,
        on="date",
        how="outer"
    )

promedios_final = promedios_final.sort_values("date")

promedios_final = promedios_final.rename(columns={
    "Precipitacion": "precip",
    "TempMax": "t_max",
    "TempMin": "t_min"
})

promedios_final["department"] = "Tolima"
promedios_final["zone"] = "Centro"

promedios_final["t_mean"] = (
    promedios_final["t_min"] + promedios_final["t_max"]
) / 2

promedios_final["source"] = "ideam"

promedios_final = promedios_final[
    [
        "department",
        "zone",
        "date",
        "t_min",
        "t_max",
        "t_mean",
        "precip",
        "source"
    ]
]

In [48]:
#Guardar nuevo escel
with pd.ExcelWriter(
    "Tolima_Completo.xlsx",
    engine="openpyxl"
) as writer:

    for nombre, df in hojas_completado.items():
        df.to_excel(
            writer,
            sheet_name=nombre,
            index=False
        )

    promedios_final.to_excel(
        writer,
        sheet_name="Promedios_diarios",
        index=False
    )

print("Archivo creado: Tolima_Completo.xlsx")

Archivo creado: Tolima_Completo.xlsx
